# 无穷维拓展检验：共同压缩能否加强 π 的有理逼近下界？

2026-11-01。研究对象：用户上传、署名 OpenAI 的《The irrationality exponent of π is 2》，以及对其机制的受控拓展。

**本轮结论：可以构造真正的无穷维对象；没有得到更强的 π 下界。现有两分支估计在指数 2 的端点存在一个对所有维数统一成立的障碍。**

这份记录区分三类内容：上传稿件中的公式；从这些公式新推导的条件性结论；为检验机制而另外定义的无穷维模型。未完成上传稿件的整篇证明审计，也不把它的主定理作为已经独立核验的事实。

这里将“更强”具体检验为：是否能证明存在固定常数 c>0，使所有整数 p、正整数 q 都满足 |π−p/q|≥c/q²。这比稿件宣称的无理性指数等于 2 更强。

所有代码单元已按顺序执行。代数证明写在正文；符号运算与高精度计算负责复核，有限计算不替代无限维证明。


## 1. 直接把原来的共同曲线放入无穷维

稿件中的共同曲线是 Γ_m(z)=(e^z,z,…,z)，具有 m 个重复的 z 坐标。直接无限重复时，它一般不属于通常的平方和空间 ℓ²。

选定一种明确的收敛结构：第 i 个方向乘以 a_i=2^(−i)，其中 i=1,2,…。在复 Hilbert 空间 ℂ⊕ℓ² 中定义

\[
\Gamma_\infty(z)=(e^z,2^{-1}z,2^{-2}z,\ldots).
\]

这是真正的无穷维解析曲线。它的距离可直接求和：

\[
\|\Gamma_\infty(z)-\Gamma_\infty(w)\|^2
=|e^z-e^w|^2+\frac13|z-w|^2.
\]

因此，无穷多个重复方向仍只提供一个独立的线性 z 方向；它们不会自动制造无限多个独立变化。

令 ω=2πi，选择有理数 p_i/q_i∈[3,4]，并令 r_i=2i p_i/q_i。对固定非负整数 j，定义

\[
\mathcal A_j=(1,j2^{-1}r_1,j2^{-2}r_2,\ldots),\qquad
\mathcal C_j=\Gamma_\infty(j\omega).
\]

每个 \(\mathcal A_j\) 坐标都在有理复数域中，且

\[
\|\mathcal A_j-\mathcal C_j\|^2
=4j^2\sum_{i\ge1}4^{-i}|p_i/q_i-\pi|^2.
\]

该和收敛。截掉第 N 个坐标之后的平方距离误差不超过 \(4j^2 4^{-N}/3\)，因为所选实数误差均小于 1。

**固定这条有理数序列时，加入更多坐标会增加、而不是减少平方距离。** 后面的误差再小，也不能消掉第一个非零误差。要使整向量趋向目标，必须让先前坐标也一起改善。这是原始有限维构造扩成一个固定无限对象后，立即可以检验的性质。这里选择的是明确的 ℓ² 几何，未宣称它是用户直觉唯一可能的形式化。


In [1]:
import math
import sympy as sp
import mpmath as mp
from fractions import Fraction
mp.mp.dps = 100

# π 的前 24 个连分数收敛分数；只作有限数值复核。
rema = mp.pi
pprev2, pprev1, qprev2, qprev1 = 0, 1, 1, 0
convergents = []
for _ in range(24):
    a = int(mp.floor(rema))
    p, q = a*pprev1+pprev2, a*qprev1+qprev2
    convergents.append((p,q))
    pprev2,pprev1,qprev2,qprev1 = pprev1,p,qprev1,q
    rema = 1/(rema-a)
convergents = convergents[1:]  # 从 22/7 开始
distances = []
for N in [1,2,4,8,16]:
    d2 = 4*mp.fsum(mp.mpf(4)**(-i)*(mp.mpf(p)/q-mp.pi)**2
                  for i,(p,q) in enumerate(convergents[:N],1))
    distances.append(mp.sqrt(d2))
    print('N=',N,'distance=',mp.nstr(mp.sqrt(d2),22),
          'squared tail bound=',mp.nstr(4*mp.mpf(4)**(-N)/3,8))
assert all(a <= b for a,b in zip(distances,distances[1:]))
print('固定前面坐标时，截面距离单调不减：通过。')


N= 1 distance= 0.001264489267349618680214 squared tail bound= 0.33333333
N= 2 distance= 0.001265173697104036162545 squared tail bound= 0.083333333
N= 4 distance= 0.001265173698861779392017 squared tail bound= 0.0052083333
N= 8 distance= 0.001265173698861779567657 squared tail bound= 2.0345052e-5
N= 16 distance= 0.001265173698861779567657 squared tail bound= 3.1044086e-10
固定前面坐标时，截面距离单调不减：通过。


## 2. 加入无限多个真正不同的幂坐标

为检验“更多独立高阶信息”是否改善误差，另取受控模型

\[
F(x)=((x/8)^k)_{k\ge1}\in\ell^2,\quad 3\le x\le4.
\]

有理输入使每个坐标都是有理数。令 s=x/8、t=y/8；几何级数直接给出

\[
\|F(x)-F(y)\|^2
=\frac{(s-t)^2(1+st)}{(1-s^2)(1-t^2)(1-st)}.
\]

在整个区间一致成立

\[
\frac{|x-y|}{8}\le\|F(x)-F(y)\|
\le\frac{\sqrt{15}}{18}|x-y|.
\]

下界来自第一个坐标。上界使用 |s^k−t^k|≤k(1/2)^(k−1)|s−t|，再求和
\(\sum_{k\ge1} k^2r^{2k-2}=(1+r^2)/(1-r^2)^3\)。

因此，该无限维距离与原始实数误差只差固定倍数，逼近指数保持不变。第 k 个有理坐标的一个可用分母是 (8q)^k；增加高阶信息时，不能继续只按原分母 q 计算算术成本。


In [2]:
s,t = sp.symbols('s t')
series_sum = s*s/(1-s*s) + t*t/(1-t*t) - 2*s*t/(1-s*t)
closed = (s-t)**2*(1+s*t)/((1-s*s)*(1-t*t)*(1-s*t))
assert sp.factor(series_sum-closed) == 0
print('无限级数距离恒等式：符号复核通过。')
for p,q in [(22,7),(355,113),(103993,33102)]:
    x=mp.mpf(p)/q
    ss,tt=x/8,mp.pi/8
    d=mp.sqrt((ss-tt)**2*(1+ss*tt)/((1-ss*ss)*(1-tt*tt)*(1-ss*tt)))
    error=abs(x-mp.pi)
    assert error/8 <= d <= mp.sqrt(15)/18*error
    print(f'{p}/{q}: error={mp.nstr(error,14)}, infinite distance={mp.nstr(d,14)}, ratio={mp.nstr(d/error,12)}')


无限级数距离恒等式：符号复核通过。
22/7: error=0.0012644892673496, infinite distance=0.00021834121066054, ratio=0.172671462145
355/113: error=2.6676418906242e-7, infinite distance=4.605625417381e-8, ratio=0.172647814295
103993/33102: error=5.7789063439038e-10, infinite distance=9.9771552039807e-11, ratio=0.172647809295


## 3. 共同体积的压缩，与分母成本同步增长

取 m≥2 个不同有理数 x_i=p_i/q_i，且 1≤q_i≤Q。令 M=m(m−1)/2，并使用前 m 个含常数项的幂坐标：

\[
D_m=\det(1,x_i/8,\ldots,(x_i/8)^{m-1})_{i=1}^m.
\]

范德蒙德恒等式给出精确关系

\[
D_m=8^{-M}\frac{\prod_{i<j}(p_jq_i-p_iq_j)}{\prod_i q_i^{m-1}}.
\]

分子为非零整数，所以 |D_m|≥8^(−M)Q^(−2M)。若所有 x_i 在长度为 L 的区间中，则 |D_m|≤8^(−M)L^M。比较得到

\[
L\ge Q^{-2}.
\]

**这组估计中的维数消去了：几何压缩的次数与分母损失的次数同步增长。** 这只是此模型、此组估计的结论，不是所有高维方法都无法加强的证明。更精细的点分布估计可以改善依赖点数的常数，不能把这里的粗界误称为最优界。

而且这个结论约束多个有理数的相互间距，并未限制单个有理数到 π 的距离。直接把 π 那一行插入行列式，会失去上述非零整数分子，需要另用 π 的特定关系恢复算术控制。

甚至不截断坐标也有精确公式：令 \(\widetilde F(t)=(1,t,t^2,\ldots)\)，其中 t 为实数且 |t|<1。其内积为 1/(1−st)。有限个点的完整无限维 Gram 行列式为

\[
\det\!\left(\frac1{1-t_it_j}\right)_{i,j=1}^m
=\frac{\prod_{i<j}(t_i-t_j)^2}
{\prod_i(1-t_i^2)\prod_{i<j}(1-t_it_j)^2}.
\]

该式已包含所有坐标，体积仍由差值乘积控制。


In [3]:
for m in range(2,7):
    xs=[sp.Rational(3)+sp.Rational(i,101) for i in range(m)]
    M=m*(m-1)//2
    D=sp.Matrix([[(x/8)**k for k in range(m)] for x in xs]).det()
    pqs=[(int(x.p),int(x.q)) for x in xs]
    integer_product=sp.prod(pqs[j][0]*pqs[i][1]-pqs[i][0]*pqs[j][1]
                            for i in range(m) for j in range(i+1,m))
    arithmetic=sp.Rational(1,8**M)*integer_product/sp.prod(q**(m-1) for p,q in pqs)
    assert D==arithmetic and D!=0
    Q=max(q for p,q in pqs)
    assert abs(D)>=sp.Rational(1,8**M*Q**(2*M))
    assert xs[-1]-xs[0]>=sp.Rational(1,Q**2)
print('m=2,…,6：行列式与整数分子恒等式精确复核通过。')
ts=[sp.Rational(3,8),sp.Rational(22,56),sp.Rational(355,904)]
gram=sp.Matrix([[1/(1-a*b) for b in ts] for a in ts])
numerator=sp.prod((ts[i]-ts[j])**2 for i in range(len(ts)) for j in range(i+1,len(ts)))
denominator=sp.prod(1-a*a for a in ts)*sp.prod((1-ts[i]*ts[j])**2 for i in range(len(ts)) for j in range(i+1,len(ts)))
assert sp.factor(gram.det()-numerator/denominator)==0
print('包含全部无限坐标的 Gram 恒等式：有理数精确复核通过。')


m=2,…,6：行列式与整数分子恒等式精确复核通过。
包含全部无限坐标的 Gram 恒等式：有理数精确复核通过。


## 4. 原稿件实际上已经使用维数趋大的收益

来源：上传稿件 §4，式 (4.4)–(4.9)，印刷页 17–19。

固定逼近指数 ν>2 后，稿件选择参数 A、B、C、θ、η；然后研究 m→∞ 时的收益与误差，取一个足够大的有限 m，再固定分母和权重，最后让多项式次数 H→∞。

它不是只尝试了某个小维度；但也没有建立一个固定无穷维物体及其统一极限。固定 ν 和 C>1，若用同一个无限分母序列的前 m 项，则最小权重 w_*≤w_1 为固定上界，而误差包含

\[
\frac{\log(2K)+\log(200K)}{w_*},\quad K=\lfloor C^m\rfloor.
\]

这项至少按 (2m log C+O(1))/w_1 发散。因此原方案增加维数后必须重新选择包括第一项在内的整批大分母，不能直接把它们当成同一无限对象的兼容截面。


## 5. 指数 2 处，对所有维数成立的现有方法障碍

以下结论是对稿件 Theorem 2.1、Proposition 3.4 及其明确误差界的推论，**仅排除这些充分条件的闭合，不排除所有可能的证明**。

记 m≥1 为维数；K 为中心个数；w_0、v_0 为正权重；0<A,θ,η<1 为截断与计数参数。令

\[
\Lambda=4\log2,\quad c=\frac{\log2}{4},\quad F_0>2/\theta,
\]
\[
E=E_{\rm ar}+E_{\rm an}>0,\qquad
L=\frac{\eta^2K\theta^m}{(m+1)v_0A^m}.
\]

E 是稿件误差上界的总和，cL 是低指标分支所保证的碰撞收益。插值条件与误差项给出

\[
v_0>K w_0\theta^m,\qquad
E\ge\frac{100K}{w_0}+\frac{\Lambda F_0m}{v_0}.
\]

由第一式替换 1/w_0，再使用两项算术—几何平均不等式，得

\[
E>20\sqrt{\Lambda F_0m}\frac{K\theta^{m/2}}{v_0}.
\]

所以对于所有允许参数都有

\[
\boxed{\frac{cL}{E}
<\frac{c\eta^2}{20(m+1)\sqrt{\Lambda F_0m}}
\left(\frac{\theta}{A^2}\right)^{m/2}.}
\]

若直接将指数取为 ν=2，高指标分支需要

\[
g=2A(1-\eta)-1-\theta>0.
\]

这迫使 θ<2A−1<A²。于是

\[
\frac{cL}{E}<\frac{c}{40\sqrt{2\Lambda}}<0.00184.
\]

而原判据需要 cL>1+E。二者不可能同时成立。参数随 m 改变也不能逃过这个逐维、统一的估计。

### 修正平均值后仍成立

稿件使用较粗的 \(\bar b\le\theta\)。行指标在一个 (m+1) 维单纯形中计数；每个归一化坐标的平均值是 1/(m+2)，因此实际有
\(\bar b_H\to\theta m/(m+2)\)。

采用精确平均值后，高指标分支要求
\(2A(1-\eta)-1-\theta m/(m+2)>0\)，从而

\[
\left(\frac\theta{A^2}\right)^{m/2}
<\left(1+\frac2m\right)^{m/2}<e.
\]

所以 cL/E<0.005001，依旧小于 1。低指标分支即使改用精确代价，也需要 cL>(1−\bar b)+E>E，仍无法闭合。

这说明障碍不仅来自 §4 的特定参数选择，也不只来自平均值的一处粗估计。它是当前收益下界、误差上界和插值条件之间的冲突。它并不证明实际误差具有如此大的不可消除下界；若重新估计，仍可能改变结论。


In [4]:
Lambda=4*mp.log(2)
c=mp.log(2)/4
bound=c/(40*mp.sqrt(2*Lambda))
print('原两分支 cL/E 统一上界：',mp.nstr(bound,20))
print('使用精确平均后统一上界：',mp.nstr(mp.e*bound,20))
assert bound < mp.mpf('0.00184')
assert mp.e*bound < mp.mpf('0.005001')
A,theta,eta=sp.symbols('A theta eta',real=True)
assert sp.expand((1+theta-2*A)-((1-A)**2+theta-A*A))==0
print('端点参数冲突的恒等式：通过。')


原两分支 cL/E 统一上界： 0.0018397031601804292047
使用精确平均后统一上界： 0.0050008316700771408785
端点参数冲突的恒等式：通过。


## 6. 接近端点时，维数增长与余量塌缩同时发生

令 ε=ν−2>0。原 §4 参数满足 θ>A²。暂不扣除 η 的高指标余量为

\[
g_0=\nu A-1-(\nu-1)\theta
<\nu A-1-(\nu-1)A^2
\le\frac{(\nu-2)^2}{4(\nu-1)}.
\]

这是配方即可验证的精确界。又因 g=g_0−νAη>0 且 νA>1，有 η<g_0=O(ε²)。

由 B²<θ 以及 g_0>0 可得

\[
\log(B/A)<\frac{\varepsilon^2}{8(1+\varepsilon)}.
\]

原方案的碰撞参数是
\(L=\eta^2(B/A)^m/[2(m+1)]\)。要使它大于一个固定正数，必要条件为

\[
m=\Omega\!\left(\varepsilon^{-2}\log\frac1\varepsilon\right).
\]

证明：η²=O(ε⁴)，而指数增长速度至多 exp(O(mε²))；对 L 的固定正下界取对数即可。此处只给必要下界，不宣称这个维数足以满足其他误差和插值阈值。

“每个 ε>0 都有某个足够大的有限维度”不提供在 ε=0 时仍有效的正余量。


In [5]:
nu,A=sp.symbols('nu A',positive=True)
quadratic=nu*A-1-(nu-1)*A*A
vertex=nu/(2*(nu-1))
best=(nu-2)**2/(4*(nu-1))
assert sp.factor(best-quadratic-(nu-1)*(A-vertex)**2)==0
for eps in [mp.mpf('1'),mp.mpf('.1'),mp.mpf('.01'),mp.mpf('.001')]:
    print('epsilon=',mp.nstr(eps,6),'max gap <',mp.nstr(eps**2/(4*(1+eps)),10),
          'log growth rate <',mp.nstr(eps**2/(8*(1+eps)),10))


epsilon= 1.0 max gap < 0.125 log growth rate < 0.0625
epsilon= 0.1 max gap < 0.002272727273 log growth rate < 0.001136363636
epsilon= 0.01 max gap < 2.475247525e-5 log growth rate < 1.237623762e-5
epsilon= 0.001 max gap < 2.497502498e-7 log growth rate < 1.248751249e-7


## 7. 再推进一步：把两种衰减联合起来优化

上一节只审计了稿件的两分支判据。本轮还检验一个自然修改：保留每项里的逼近误差衰减与碰撞衰减，不先按阈值分开。以下是固定 m 后、H→∞ 的连续松弛问题，不是声称已得到对 m、H 联合极限一致的定理。

记 R=Kθ^m/v_0。类型 a 的归一化权重为 t=wa/H∈[0,1]，类型密度按 m t^(m−1)dt 分布。令 Hf(t) 描述该类型的占用次数。暂时放松整数性和每一行 a≥β 的限制，则联合收益的渐近最小值是

\[
F_{\min}=\min_{f\ge0}\frac{m+1}{R}\int_0^1
\bigl(\nu t f(t)+cf(t)^2\bigr)m t^{m-1}\,dt,
\]
\[
\int_0^1 f(t)m t^{m-1}\,dt=\frac R{m+1}.
\]

其中 c=log(2)/4，第一项来自逼近误差，第二项来自重复阶数碰撞。逐点凸优化给出 f(t)=(λ−νt)_+/(2c)。类型格点的黎曼和以及占用数取整，使此式成为放松问题在固定 m 下的渐近最优值。

**内部支持：R≤ν/(2c)。** 令 T=(2cR/ν)^(1/(m+1))≤1，则

\[
f(t)=\frac{\nu}{2c}(T-t)_+,\qquad
F_{\min}=\nu T\frac{m+1}{m+2}.
\]

记 b=θm/(m+2) 为精确行平均值。要与算术下界矛盾，需要 F_min>1+(ν−1)b+E。若 ν=2，即使先忽略 E，也必须

\[
T>\frac{m+2+m\theta}{2(m+1)}>\sqrt\theta.
\]

最后一步由 m(1−√θ)²+2(1−√θ)>0 得到。但现有总误差界的数值满足

\[
E>20\sqrt{\Lambda F_0m}\,R\theta^{-m/2}
=\frac{20}{c}\sqrt{\Lambda F_0m}\,T(T/\sqrt\theta)^m
>2T>F_{\min}.
\]

**支持饱和：R≥ν/(2c)。** 此时最优函数在整个 [0,1] 上非负，λ=(νm+2cR)/(m+1)，积分得到

\[
F_{\min}=\frac{cR}{m+1}+\frac{\nu m}{m+1}
-\frac{\nu^2m}{4cR(m+1)(m+2)}.
\]

在 ν=2 时 R≥1/c，因此 F_min≤2.5cR，而 E>20√(2Λ)R>47R，仍有 E>F_min。

**结论：这个自然的联合松弛估计也不能在端点闭合。** 不是因为两类收益没有相加，而是相加并优化后仍受现有成本估计限制。这里放松了每一行允许出现哪些类型的对应关系；若保留该关系，或者改进误差界，结论可能不同，本轮没有排除。


In [6]:
t,R,c0,nu=sp.symbols('t R c nu',positive=True)
for m in range(1,7):
    lam=(nu*m+2*c0*R)/(m+1)
    f=(lam-nu*t)/(2*c0)
    mass=sp.integrate(f*m*t**(m-1),(t,0,1))
    cost=(m+1)/R*sp.integrate((nu*t*f+c0*f*f)*m*t**(m-1),(t,0,1))
    expected=c0*R/(m+1)+nu*m/(m+1)-nu**2*m/(4*c0*R*(m+1)*(m+2))
    assert sp.factor(mass-R/(m+1))==0
    assert sp.factor(cost-expected)==0
    T=sp.symbols('T',positive=True)
    f_inner=nu*(T-t)/(2*c0)
    mass_inner=sp.integrate(f_inner*m*t**(m-1),(t,0,T))
    expected_mass=nu*T**(m+1)/(2*c0*(m+1))
    assert sp.factor(mass_inner-expected_mass)==0
    cost_inner=sp.integrate((nu*t*f_inner+c0*f_inner*f_inner)*m*t**(m-1),(t,0,T))/mass_inner
    assert sp.factor(cost_inner-nu*T*(m+1)/(m+2))==0
print('联合松弛：m=1,…,6，两种支持区间的质量与最小值公式符号复核通过。')


联合松弛：m=1,…,6，两种支持区间的质量与最小值公式符号复核通过。


## 8. 为什么无限阶抵消可能把信息也一起去掉

本节在实 Hilbert 空间中讨论。仍用 G(t)=(t,t²,…)∈ℓ²，实参数满足 |t|<1。固定 t_0∈(−1,1)，将差向量 G(t)−G(t_0) 投影到前 N 阶导数向量张成空间的正交补，余量在 t→t_0 时为 O(|t−t_0|^(N+1))。

这确实产生任意高阶的小量。但是所有正阶导数向量的张成在 ℓ² 中稠密：若 v 与它们全都正交，解析函数 g(z)=Σv_k z^k 在 t_0 的所有正阶导数均为零，因而是常数；又 g(0)=0，故 v=0。

所以 N→∞ 后，对任意固定差向量，投影余量都会趋于零。这种“无限压缩”同时抹掉了全部位置差异，不能作为原始误差变小的证明。

同理，det((1/2)I_m)=2^(−m)→0，但各方向始终正交。增加维数时，裸行列式趋零可以只是尺度乘积的效果，必须与归一化和算术成本一起解释。


## 9. 真正的无穷维行列式需要什么

Bornemann 的原始论文 §3、§5 给出标准框架：对迹类算子 T，若有限维正交投影 P_m 强收敛到恒等算子，则 det(I+zP_mTP_m) 收敛到 Fredholm 行列式 det(I+zT)，对有界 z 一致。采用嵌套且并集稠密的有限维子空间可保证投影条件。

这提供合法的分析极限。它不自动保留“清分母后的非零整数”这一算术下界，也不自动保证极限行列式非零。要将上传稿件改写到这一框架，仍需构造统一算子、验证相应收敛，并重新证明与分母成本兼容的非零估计。本轮没有完成这个重构。

## 10. 指数 2 与统一平方下界的区别

标准连分数公式为 μ(α)=2+limsup log(a_(n+1))/log(q_n)；坏逼近等价于部分商 a_n 有界。取 α=[0;1,2,3,4,…]，则 a_n 无界，而 q_n≥n!，故上述比值趋零。因此 μ(α)=2，却不存在统一正的 c/q² 下界。

所以即便上传稿件的结论最终成立，也不能仅用“让 ν 下降到 2”得到用户目标。若反设 q²|π−p/q|→0，可能多出常数因子的收益，但其速度可能比任何 q 的固定正幂都慢；需要另外跟踪，不能直接套固定 ν>2 的估计。

具体地，若 |π−p_i/q_i|≤ρ_i q_i^(−2)，令 e_i=−log ρ_i。精确行展开中，第 i 个逼近误差因子每出现一次，就额外贡献 e_i 的指数衰减；必须同时控制对数截断的尾项，不能直接把全部高阶权重都计成此种收益。上文的纯 ν=2 端点检验没有使用这些 e_i。坏逼近失败只要求 ρ_i→0；允许例如 ρ_i=1/log q_i，此时 e_i/log q_i→0。因而不能把这项统一换成某个固定的 ν>2。要真正检验坏逼近性，需要保留逐坐标的 e_i、逐行可选类型的约束以及尾项成本。

## 11. 本轮完成与未完成

- 完成了原共同曲线的一个明确无穷维扩展、收敛与尾项控制。
- 完成了包含所有幂坐标的无限维距离和 Gram 体积检验。
- 证明了受控模型中几何压缩与分母成本同步的精确公式。
- 证明了上传稿件现有两分支判据在指数 2 处对所有维数的障碍，并检查了一处自然的平均值改进。
- 完成了将两类衰减合并的连续占用数松弛优化；在两个支持区间均未突破现有端点障碍。
- 未证明 π 坏逼近；未证明所有无穷维方法都无效；未独立审计上传稿件全部证明。

接下来需要的新结构应使独立约束增长，而分母成本与分析误差不同比例增长。单纯增加幂坐标、重复坐标或投掉更多阶数，这三种操作已在本轮得到具体检验。

## 来源

1. 用户上传 paper-1.pdf：《The irrationality exponent of π is 2》，署名 OpenAI，标注日期 2026-09-24。重点使用 Theorem 2.1、§3.1、§3.3、Proposition 3.4、§4。该文件是本轮待拓展的原始稿件，不代表对其来源或主定理完成独立确认。
2. Folkmar Bornemann, *On the Numerical Evaluation of Fredholm Determinants*, §§3–5, https://arxiv.org/pdf/0804.2543 。仅用于无穷维行列式的标准收敛框架。
3. A. N. W. Hone and J. L. Varona, *Continued fractions and irrationality exponents for modified Engel and Pierce series*, §3, (3.3), https://doi.org/10.1007/s00605-018-1244-1 。用于标准无理性指数公式。
4. W. M. Schmidt, *On badly approximable numbers*, Mathematika 12 (1965), 10–20, https://www.cambridge.org/core/journals/mathematika/article/abs/on-badly-approximable-numbers/85CDAE1BE543B25DB77BAB771A16792F 。用于坏逼近与有界部分商的等价。
